# Preprocessing

This notebook builds a simple model-ready dataset from the cleaned generation and emissions tables.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

generation = pd.read_csv(PROCESSED_DIR / "generation_by_fuel_hourly.csv")
emissions = pd.read_csv(PROCESSED_DIR / "emissions_factors_by_fuel.csv")

generation.head()

,timestamp,region,region_name,fuel_type,generation_mwh
0,2024-01-01T00,PJM,"PJM Interconnection, LLC",Coal,14814
1,2024-01-01T00,PJM,"PJM Interconnection, LLC",Natural Gas,40727
2,2024-01-01T00,PJM,"PJM Interconnection, LLC",Nuclear,33462
3,2024-01-01T00,PJM,"PJM Interconnection, LLC",Petroleum,210
4,2024-01-01T00,PJM,"PJM Interconnection, LLC",Other,1008


## Merge generation with emissions factors

In [3]:
df = generation.merge(emissions, on="fuel_type", how="left")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["co2_kg"] = df["generation_mwh"] * df["kgco2e_per_mwh"]

df.head()

,timestamp,region,region_name,fuel_type,generation_mwh,kgco2e_per_mwh,co2_kg
0,2024-01-01,PJM,"PJM Interconnection, LLC",Coal,14814,1000.0,14814000.0
1,2024-01-01,PJM,"PJM Interconnection, LLC",Natural Gas,40727,450.0,18327150.0
2,2024-01-01,PJM,"PJM Interconnection, LLC",Nuclear,33462,12.0,401544.0
3,2024-01-01,PJM,"PJM Interconnection, LLC",Petroleum,210,780.0,163800.0
4,2024-01-01,PJM,"PJM Interconnection, LLC",Other,1008,NaN,NaN


## Compute hourly carbon intensity by region

In [4]:
hourly = (
    df.groupby(["timestamp", "region", "region_name"], as_index=False)
    .agg(
        total_generation_mwh=("generation_mwh", "sum"),
        total_co2_kg=("co2_kg", "sum")
    )
)

hourly["carbon_intensity_kg_per_mwh"] = (
    hourly["total_co2_kg"] / hourly["total_generation_mwh"]
)

hourly.head()

,timestamp,region,region_name,total_generation_mwh,total_co2_kg,carbon_intensity_kg_per_mwh
0,2024-01-01 00:00:00,CISO,California Independent System Operator,18740,4984996.0,266.008324
1,2024-01-01 00:00:00,NYIS,New York Independent System Operator,15626,3911152.0,250.297709
2,2024-01-01 00:00:00,PJM,"PJM Interconnection, LLC",99219,33864465.0,341.310283
3,2024-01-01 01:00:00,CISO,California Independent System Operator,18479,5264795.0,284.906921
4,2024-01-01 01:00:00,NYIS,New York Independent System Operator,15100,3758468.0,248.905166


## Add simple time features

In [5]:
hourly = hourly.sort_values(["region", "timestamp"]).reset_index(drop=True)

hourly["hour"] = hourly["timestamp"].dt.hour
hourly["day_of_week"] = hourly["timestamp"].dt.dayofweek
hourly["month"] = hourly["timestamp"].dt.month

hourly.head()

,timestamp,region,region_name,total_generation_mwh,total_co2_kg,carbon_intensity_kg_per_mwh,hour,day_of_week,month
0,2024-01-01 00:00:00,CISO,California Independent System Operator,18740,4984996.0,266.008324,0,0,1
1,2024-01-01 01:00:00,CISO,California Independent System Operator,18479,5264795.0,284.906921,1,0,1
2,2024-01-01 02:00:00,CISO,California Independent System Operator,19185,5640694.0,294.015846,2,0,1
3,2024-01-01 03:00:00,CISO,California Independent System Operator,21143,5844820.0,276.442321,3,0,1
4,2024-01-01 04:00:00,CISO,California Independent System Operator,20261,5768703.0,284.719560,4,0,1


## Add lag features

In [6]:
hourly["lag_1"] = hourly.groupby("region")["carbon_intensity_kg_per_mwh"].shift(1)
hourly["lag_24"] = hourly.groupby("region")["carbon_intensity_kg_per_mwh"].shift(24)

hourly.head(30)

,timestamp,region,region_name,total_generation_mwh,total_co2_kg,carbon_intensity_kg_per_mwh,hour,day_of_week,month,lag_1,lag_24
0,2024-01-01 00:00:00,CISO,California Independent System Operator,18740,4984996.0,266.008324,0,0,1,NaN,NaN
1,2024-01-01 01:00:00,CISO,California Independent System Operator,18479,5264795.0,284.906921,1,0,1,266.008324,NaN
2,2024-01-01 02:00:00,CISO,California Independent System Operator,19185,5640694.0,294.015846,2,0,1,284.906921,NaN
3,2024-01-01 03:00:00,CISO,California Independent System Operator,21143,5844820.0,276.442321,3,0,1,294.015846,NaN
4,2024-01-01 04:00:00,CISO,California Independent System Operator,20261,5768703.0,284.719560,4,0,1,276.442321,NaN
5,2024-01-01 05:00:00,CISO,California Independent System Operator,19602,5700629.0,290.818743,5,0,1,284.719560,NaN
6,2024-01-01 06:00:00,CISO,California Independent System Operator,18847,5716966.0,303.335597,6,0,1,290.818743,NaN
7,2024-01-01 07:00:00,CISO,California Independent System Operator,18614,5652693.0,303.679650,7,0,1,303.335597,NaN
8,2024-01-01 08:00:00,CISO,California Independent System Operator,17927,5738571.0,320.107715,8,0,1,303.679650,NaN
9,2024-01-01 09:00:00,CISO,California Independent System Operator,17478,5718790.0,327.199336,9,0,1,320.107715,NaN


## Create a day-ahead target

In [7]:
hourly["target_carbon_intensity_24h"] = (
    hourly.groupby("region")["carbon_intensity_kg_per_mwh"].shift(-24)
)

hourly.head(30)

,timestamp,region,region_name,total_generation_mwh,total_co2_kg,carbon_intensity_kg_per_mwh,hour,day_of_week,month,lag_1,lag_24,target_carbon_intensity_24h
0,2024-01-01 00:00:00,CISO,California Independent System Operator,18740,4984996.0,266.008324,0,0,1,NaN,NaN,210.280762
1,2024-01-01 01:00:00,CISO,California Independent System Operator,18479,5264795.0,284.906921,1,0,1,266.008324,NaN,210.642303
2,2024-01-01 02:00:00,CISO,California Independent System Operator,19185,5640694.0,294.015846,2,0,1,284.906921,NaN,269.147100
3,2024-01-01 03:00:00,CISO,California Independent System Operator,21143,5844820.0,276.442321,3,0,1,294.015846,NaN,276.991409
4,2024-01-01 04:00:00,CISO,California Independent System Operator,20261,5768703.0,284.719560,4,0,1,276.442321,NaN,278.731827
5,2024-01-01 05:00:00,CISO,California Independent System Operator,19602,5700629.0,290.818743,5,0,1,284.719560,NaN,284.938136
6,2024-01-01 06:00:00,CISO,California Independent System Operator,18847,5716966.0,303.335597,6,0,1,290.818743,NaN,297.821549
7,2024-01-01 07:00:00,CISO,California Independent System Operator,18614,5652693.0,303.679650,7,0,1,303.335597,NaN,295.402465
8,2024-01-01 08:00:00,CISO,California Independent System Operator,17927,5738571.0,320.107715,8,0,1,303.679650,NaN,315.895675
9,2024-01-01 09:00:00,CISO,California Independent System Operator,17478,5718790.0,327.199336,9,0,1,320.107715,NaN,322.052465


## Save the model-ready dataset

In [8]:
model_data = hourly.dropna(subset=["lag_1", "lag_24", "target_carbon_intensity_24h"]).copy()
model_data.to_csv(PROCESSED_DIR / "model_ready_data.csv", index=False)

model_data.shape

(360, 12)